<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/LLM/LLM02/HuggingFace_%E1%84%91%E1%85%B3%E1%84%85%E1%85%A9%E1%84%8C%E1%85%A6%E1%86%A8%E1%84%90%E1%85%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HuggingFace 커스텀 프로젝트 — NSMC


In [1]:
%pip install -q "transformers>=5,<6" "datasets>=4,<6" accelerate scikit-learn

## 라이브러리 버전을 확인해 봅니다.
---
사용할 라이브러리 버전을 둘러봅시다.

In [2]:
import torch
import numpy
import transformers
import datasets

print(torch.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(datasets.__version__)

2.11.0+cu128
2.1.3
5.16.1
4.8.5


In [3]:
import gc
import csv
import numpy as np
import pandas as pd
from dataclasses import replace
from IPython.display import display
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, set_seed,
)

SEED = 42
set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA 없음")

GPU: Tesla T4


## STEP 1. `NSMC` 데이터 분석 및 Huggingface dataset 구성
---
  - 데이터셋은 [깃허브(e9t/nsmc)](https://github.com/e9t/nsmc)에서 `ratings_train.txt`·`ratings_test.txt` 를 내려받거나, [Huggingface datasets 의 e9t/nsmc](https://huggingface.co/datasets/e9t/nsmc)에서 `load_dataset("e9t/nsmc")` 로 가져올 수 있습니다. 앞에서 배운 두 방법(불러오기·직접 가공)을 모두 써 보세요.

### 1-1. Hugging Face에서 불러오기

예제의 데이터셋 이름을 NSMC로 바꾸고 과제에 제시된 Parquet 변환 브랜치를 지정한다.

In [4]:
huggingface_nsmc_dataset = load_dataset("e9t/nsmc", revision="refs/convert/parquet")
print(huggingface_nsmc_dataset)
train = huggingface_nsmc_dataset['train']
cols = train.column_names
print(cols)
for i in range(3):
    for col in cols:
        print(col, ":", train[col][i])
    print()

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 11.1MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.71MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})
['id', 'document', 'label']
id : 9976970
document : 아 더빙.. 진짜 짜증나네요 목소리
label : 0

id : 3819312
document : 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
label : 1

id : 10265843
document : 너무재밓었다그래서보는것을추천한다
label : 0



### 1-2. TXT를 직접 읽어 Dataset 구성하기

예제의 TXT→DataFrame→Dataset 흐름을 사용한다. NSMC는 `id`, `document`, `label` 세 열이며 탭으로 구분된다.
아래 셀은 GitHub의 TXT를 메모리로 읽는다. 리뷰 안의 따옴표와 `NA` 같은 실제 문자열을 그대로 보존한다.

In [5]:
url = 'https://raw.githubusercontent.com/e9t/nsmc/master/'
train_df = pd.read_csv(url + 'ratings_train.txt', sep='\t', quoting=csv.QUOTE_NONE, keep_default_na=False)
test_df = pd.read_csv(url + 'ratings_test.txt', sep='\t', quoting=csv.QUOTE_NONE, keep_default_na=False)

customized_nsmc_raw = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False),
})
print(customized_nsmc_raw)
display(train_df.head())

# 빈 문자열과 결측 표현만 맞춘 뒤 리뷰·정답의 행 순서까지 비교한다.
for split, frame in [('train', train_df), ('test', test_df)]:
    hub = huggingface_nsmc_dataset[split].to_pandas()
    same_text = hub['document'].fillna('').tolist() == frame['document'].fillna('').tolist()
    same_label = hub['label'].tolist() == frame['label'].tolist()
    print(split, 'Hub/TXT 행 수:', len(hub), len(frame), '리뷰 일치:', same_text, '라벨 일치:', same_label)

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


train Hub/TXT 행 수: 150000 150000 리뷰 일치: False 라벨 일치: True
test Hub/TXT 행 수: 50000 50000 리뷰 일치: False 라벨 일치: True


### 1-3. 데이터 분석과 정제

결측·빈 리뷰를 제거하고 앞뒤 공백을 정리한다. 감정 단서가 되는 문장부호·이모티콘은 유지한다.
원본 train에서 같은 리뷰에 서로 다른 정답이 달린 경우를 제외하고 중복 리뷰는 하나만 남긴다.
이후 원본 train 안에서만 분할하므로 train/validation에 같은 리뷰가 섞이지 않는다.
test에는 결측·빈 리뷰 처리만 적용하고, 정제한 test의 점수임을 명시한다.

In [6]:
for name, frame in [('train', train_df), ('test', test_df)]:
    print(name, '행 수:', len(frame), '결측:', frame['document'].isna().sum(),
          '빈 리뷰:', frame['document'].fillna('').str.strip().eq('').sum(),
          '중복:', frame['document'].duplicated().sum())
    print(frame['label'].value_counts(normalize=True).sort_index())
display(train_df['document'].str.len().describe().to_frame('글자 수'))

def clean_data(df):
    df = df.dropna(subset=['document', 'label']).copy()
    df['document'] = df['document'].str.strip()
    return df.loc[df['document'].ne(''), ['document', 'label']].reset_index(drop=True)

train_df_hf = clean_data(train_df)
test_df_hf = clean_data(test_df)
conflicts = train_df_hf.groupby('document')['label'].nunique()
train_df_hf = train_df_hf[~train_df_hf['document'].isin(conflicts[conflicts > 1].index)]
train_df_hf = train_df_hf.drop_duplicates(subset='document').reset_index(drop=True)
print('정제 전후 train:', len(train_df), '→', len(train_df_hf))
print('정제 전후 test:', len(test_df), '→', len(test_df_hf))

train 행 수: 150000 결측: 0 빈 리뷰: 5 중복: 3817
label
0    0.501153
1    0.498847
Name: proportion, dtype: float64
test 행 수: 50000 결측: 0 빈 리뷰: 3 중복: 842
label
0    0.49654
1    0.50346
Name: proportion, dtype: float64


,글자 수
count,150000.000000
mean,35.237393
std,29.582475
min,0.000000
25%,16.000000
50%,27.000000
75%,42.000000
max,158.000000


정제 전후 train: 150000 → 146025
정제 전후 test: 50000 → 49997


In [7]:
# 첨부 예제의 분할 비율·시드·층화를 유지한다.
train_split, val_split = train_test_split(
    train_df_hf, test_size=0.2, random_state=42, stratify=train_df_hf['label'])

customized_nsmc_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_split, preserve_index=False),
    'validation': Dataset.from_pandas(val_split, preserve_index=False),
    'test': Dataset.from_pandas(test_df_hf, preserve_index=False),
})
print(customized_nsmc_dataset)
display(pd.DataFrame({
    name: frame['label'].value_counts().sort_index()
    for name, frame in [('train', train_split), ('validation', val_split), ('test', test_df_hf)]
}).rename(index={0: '부정', 1: '긍정'}))
overlap = set(train_split['document']) & set(val_split['document'])
print('train/validation 중복 리뷰:', len(overlap))
assert not overlap

DatasetDict({
    train: Dataset({
        features: ['document', 'label'],
        num_rows: 116820
    })
    validation: Dataset({
        features: ['document', 'label'],
        num_rows: 29205
    })
    test: Dataset({
        features: ['document', 'label'],
        num_rows: 49997
    })
})


,train,validation,test
label,,,
부정,58618,14654,24826
긍정,58202,14551,25171


train/validation 중복 리뷰: 0


## STEP 2. klue/bert-base model 및 tokenizer 불러오기
---

In [8]:
set_seed(SEED)
huggingface_tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')
huggingface_model = AutoModelForSequenceClassification.from_pretrained('klue/bert-base', num_labels=2)

sample = huggingface_tokenizer('정말 재미있는 영화였어요.')
print(sample)
print(huggingface_tokenizer.convert_ids_to_tokens(sample['input_ids']))

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'input_ids': [2, 3944, 6001, 2259, 3771, 2507, 10283, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', '정말', '재미있', '##는', '영화', '##였', '##어요', '.', '[SEP]']


## STEP 3. 위에서 불러온 tokenizer으로 데이터셋을 전처리하고, model 학습 진행해 보기
---

In [9]:
# 정제·분할을 마친 train 전체를 자르지 않고 토큰화한다. 특수 토큰을 포함한 길이이다.
train_texts = customized_nsmc_dataset['train']['document']
lengths = []
for start in range(0, len(train_texts), 1000):
    tokens = huggingface_tokenizer(
        train_texts[start:start + 1000], truncation=False, padding=False,
        add_special_tokens=True, return_token_type_ids=False,
    )
    lengths.extend(len(ids) for ids in tokens['input_ids'])
lengths = np.array(lengths)
print('길이를 확인한 훈련 리뷰 수:', len(lengths))
display(pd.Series(lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame('토큰 수'))

# 훈련 리뷰를 최대한 보존하되 모델과 토크나이저의 허용 길이를 넘지 않는다.
model_limit = min(huggingface_tokenizer.model_max_length, huggingface_model.config.max_position_embeddings)
MAX_LENGTH = min(int(lengths.max()), model_limit)
candidates = sorted({min(length, model_limit) for length in [32, 64, 128, 256, MAX_LENGTH, model_limit]})
length_comparison = pd.DataFrame({
    '최대 길이': candidates,
    '잘리는 리뷰 수': [int((lengths > length).sum()) for length in candidates],
    '잘리는 리뷰 비율(%)': [float((lengths > length).mean() * 100) for length in candidates],
})
display(length_comparison)
print('훈련 데이터 최대 토큰 길이:', int(lengths.max()))
print('모델·토크나이저 허용 길이:', model_limit)
print('모든 실험의 MAX_LENGTH:', MAX_LENGTH)
print(f'선택한 길이에서 잘리는 훈련 리뷰 비율: {(lengths > MAX_LENGTH).mean():.2%}')

def transform(data):
    return huggingface_tokenizer(
        data['document'],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
        return_token_type_ids=False,
    )

custom_dataset = customized_nsmc_dataset.map(transform, batched=True, remove_columns=['document'])
custom_train_dataset = custom_dataset['train']
custom_val_dataset = custom_dataset['validation']
custom_test_dataset = custom_dataset['test']
data_collator = DataCollatorWithPadding(tokenizer=huggingface_tokenizer)

길이를 확인한 훈련 리뷰 수: 116820


,토큰 수
count,116820.000000
mean,22.790849
std,16.752629
min,3.000000
50%,18.000000
90%,45.000000
95%,63.000000
99%,81.000000
max,142.000000


,최대 길이,잘리는 리뷰 수,잘리는 리뷰 비율(%)
0,32,20399,17.461907
1,64,5501,4.708954
2,128,2,0.001712
3,142,0,0.000000
4,256,0,0.000000
5,512,0,0.000000


훈련 데이터 최대 토큰 길이: 142
모델·토크나이저 허용 길이: 512
모든 실험의 MAX_LENGTH: 142
선택한 길이에서 잘리는 훈련 리뷰 비율: 0.00%


Map:   0%|          | 0/116820 [00:00<?, ? examples/s]

Map:   0%|          | 0/29205 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

In [10]:
batch = data_collator([custom_train_dataset[i] for i in range(2)])
print({key: (tuple(value.shape), str(value.dtype)) for key, value in batch.items()})
huggingface_model.eval()
with torch.no_grad():
    output = huggingface_model(**batch)
print('logits:', tuple(output.logits.shape), 'loss:', output.loss.item())
del batch, output

{'input_ids': ((2, 12), 'torch.int64'), 'attention_mask': ((2, 12), 'torch.int64'), 'labels': ((2,), 'torch.int64')}
logits: (2, 2) loss: 0.6225640773773193


### 3-2. 파인튜닝 전 평가와 기본 설정 파인튜닝

`TrainingArguments`와 `Trainer`를 예제처럼 구성한다.
먼저 `trainer.evaluate()`로 NSMC 학습 전 검증 정확도를 기록한다. 이때 감정 분류층은 새로 초기화된 상태이다.
이어서 `trainer.train()`으로 기본 설정의 파인튜닝을 수행하고, 같은 validation에서 학습 전후 정확도를 비교한다.
각 에포크의 검증 정확도가 가장 높은 체크포인트를 선택한다.

측정값 `train_runtime`과 `train_samples_per_second`는 `Trainer.train()`이 보고한 값이다.
에포크 중 평가·체크포인트 저장 비용이 포함되며, 앞에서 수행한 다운로드·토큰화·학습 전 평가는 제외된다.
①은 평가만 수행하므로 학습 시간·처리량 칸은 비워 둔다.

In [11]:
output_dir = './results/nsmc_step3'
training_arguments = TrainingArguments(
    output_dir,
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    train_sampling_strategy='random',
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy': float(np.mean(predictions == labels))}

# 학습한 세 모델에서 같은 항목을 기록한다.
def result_row(name, trainer, train_output, max_length):
    evaluation = trainer.evaluate()
    return {
        'experiment': name,
        'max_length': max_length,
        'sampling': trainer.args.train_sampling_strategy,
        'learning_rate': trainer.args.learning_rate,
        'epochs': trainer.args.num_train_epochs,
        'val_accuracy': evaluation['eval_accuracy'],
        'train_seconds': train_output.metrics['train_runtime'],
        'samples_per_second': train_output.metrics['train_samples_per_second'],
        'checkpoint': trainer.state.best_model_checkpoint,
    }

In [12]:
trainer = Trainer(
    model=huggingface_model,
    args=training_arguments,
    train_dataset=custom_train_dataset,
    eval_dataset=custom_val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)
# ① 파인튜닝 전: 아직 NSMC를 학습하지 않은 분류기를 같은 validation으로 평가한다.
before_evaluation = trainer.evaluate()
before_result = {
    'experiment': '① 파인튜닝 전',
    'max_length': MAX_LENGTH,
    'sampling': '평가만 수행',
    'learning_rate': None,
    'epochs': 0,
    'val_accuracy': before_evaluation['eval_accuracy'],
    'train_seconds': None,
    'samples_per_second': None,
    'checkpoint': None,
}
display(pd.DataFrame([before_result]))

# ② 기본 설정으로 파인튜닝.
set_seed(SEED)
train_output = trainer.train()
baseline_result = result_row('② 기본 파인튜닝', trainer, train_output, MAX_LENGTH)
display(pd.DataFrame([before_result, baseline_result]))
change = (baseline_result['val_accuracy'] - before_result['val_accuracy']) * 100
print(f'파인튜닝 전후 정확도 변화: {change:+.2f} 퍼센트포인트')

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.687202,0,0.569115


,experiment,max_length,sampling,learning_rate,epochs,val_accuracy,train_seconds,samples_per_second,checkpoint
0,① 파인튜닝 전,142,평가만 수행,None,0,0.569115,None,None,None


Epoch,Training Loss,Validation Loss,Accuracy
1,0.287146,0.309708,0.900360
2,0.215140,0.378718,0.908201
3,0.128696,0.462020,0.907379


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.128696,0.378718,3,0.908201


,experiment,max_length,sampling,learning_rate,epochs,val_accuracy,train_seconds,samples_per_second,checkpoint
0,① 파인튜닝 전,142,평가만 수행,NaN,0,0.569115,NaN,NaN,None
1,② 기본 파인튜닝,142,random,0.00002,3,0.908201,4534.6479,77.285,./results/nsmc_step3/checkpoint-29206


파인튜닝 전후 정확도 변화: +33.91 퍼센트포인트


## STEP 4. Fine-tuning을 통하여 모델 성능(accuracy) 향상시키기
  - 데이터 전처리, `TrainingArguments` 등을 조정하여 모델의 정확도를 90% 이상으로 끌어올려봅시다.

**개선 가설:** 학습률을 2e-5→3e-5로 높이면 같은 3에포크 동안 NSMC에 더 잘 적응할 수 있다.
검증 손실 증가나 정확도 하락 가능성도 있어, 3e-5는 효과를 확인할 후보 설정이다.
훈련 데이터로 결정한 최대 길이·배치 8·에포크 3·분할·시드·동적 패딩은 유지한다.
먼저 기본 모델의 오분류 5개를 확인하고, 같은 사전학습 모델을 새로 불러와 학습한다.
②에서 학습한 모델을 이어 학습하지 않으므로 학습률 변경 효과를 같은 학습량에서 비교할 수 있다.

In [13]:
prediction = trainer.predict(custom_val_dataset)
predicted_labels = np.argmax(prediction.predictions, axis=1)
errors = val_split.reset_index(drop=True).copy()
errors['prediction'] = predicted_labels
display(errors.loc[errors['label'] != errors['prediction']].head())
del prediction

# 첨부 예제의 메모리 정리와 모델 재생성 코드를 재사용한다.
del trainer, huggingface_model
gc.collect()
torch.cuda.empty_cache()

set_seed(SEED)
huggingface_model = AutoModelForSequenceClassification.from_pretrained('klue/bert-base', num_labels=2)
# ③ 최대 길이·배치·에포크를 유지하고 학습률 한 가지만 바꾼다.
improved_arguments = replace(
    training_arguments,
    output_dir='./results/nsmc_step4',
    learning_rate=3e-5,
)

,document,label,prediction
7,내가 이 영화를 100% 다 소화시킬수 없다는 것에 정말 슬프다...,1,0
9,글로는 힘들어~~ㅋㅋ,0,1
16,북한 문제도 그렇고 007 시리즈를 리부트로 내모는데 기여한 작품..,0,1
18,나도 30살 연하의 아가씨가 밑고 끝도없이 대쉬해줬으면 좋겠다.,0,1
20,신선한 충격과 국민의 수준 역시 저급하다는걸 느꼈습니다..,1,0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
trainer = Trainer(
    model=huggingface_model,
    args=improved_arguments,
    train_dataset=custom_train_dataset,
    eval_dataset=custom_val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)
train_output = trainer.train()
improved_result = result_row('③ 학습률 변경', trainer, train_output, MAX_LENGTH)
display(pd.DataFrame([before_result, baseline_result, improved_result]))
change = (improved_result['val_accuracy'] - baseline_result['val_accuracy']) * 100
print(f'기준 대비 정확도 변화: {change:+.2f} 퍼센트포인트')
print('STEP 4 검증 정확도 90%:', '달성' if improved_result['val_accuracy'] >= 0.90 else '미달')

Epoch,Training Loss,Validation Loss,Accuracy
1,0.315464,0.332265,0.897928
2,0.242581,0.368890,0.902003
3,0.157629,0.446259,0.903818


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.157629,0.446259,3,0.903818


,experiment,max_length,sampling,learning_rate,epochs,val_accuracy,train_seconds,samples_per_second,checkpoint
0,① 파인튜닝 전,142,평가만 수행,NaN,0,0.569115,NaN,NaN,None
1,② 기본 파인튜닝,142,random,0.00002,3,0.908201,4534.6479,77.285,./results/nsmc_step3/checkpoint-29206
2,③ 학습률 변경,142,random,0.00003,3,0.903818,4583.3062,76.464,./results/nsmc_step4/checkpoint-43809


기준 대비 정확도 변화: -0.44 퍼센트포인트
STEP 4 검증 정확도 90%: 달성


## STEP 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와 비교



In [15]:
del trainer, huggingface_model
gc.collect()
torch.cuda.empty_cache()

set_seed(SEED)
huggingface_model = AutoModelForSequenceClassification.from_pretrained('klue/bert-base', num_labels=2)
bucket_arguments = replace(
    improved_arguments,
    output_dir='./results/nsmc_step5_bucket',
    train_sampling_strategy='group_by_length',
)
trainer = Trainer(
    model=huggingface_model,
    args=bucket_arguments,
    train_dataset=custom_train_dataset,
    eval_dataset=custom_val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)
train_output = trainer.train()
bucket_result = result_row('④ 버케팅 파인튜닝', trainer, train_output, MAX_LENGTH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.329520,0.299463,0.898921
2,0.253805,0.408208,0.901181
3,0.139944,0.457949,0.902859


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.139944,0.457949,3,0.902859


In [16]:
comparison = pd.DataFrame([before_result, baseline_result, improved_result, bucket_result])
display(comparison)
speed_change = (1 - bucket_result['train_seconds'] / improved_result['train_seconds']) * 100
accuracy_change = (bucket_result['val_accuracy'] - improved_result['val_accuracy']) * 100
print(f'STEP 4 대비 학습 시간 절감률: {speed_change:+.2f}%')
print(f'STEP 4 대비 검증 정확도 변화: {accuracy_change:+.2f} 퍼센트포인트')
if speed_change > 0 and accuracy_change < 0:
    print('이번 실행에서는 학습 시간이 줄고 정확도가 낮아지는 trade-off(상충관계)가 관찰되었다.')
elif speed_change < 0 and accuracy_change > 0:
    print('이번 실행에서는 정확도가 높아지고 학습 시간이 늘어나는 trade-off(상충관계)가 관찰되었다.')
elif speed_change > 0 and accuracy_change >= 0:
    print('이번 실행에서는 학습 시간이 줄고 정확도는 유지되거나 높아졌다.')
else:
    print('이번 실행에서는 버케팅의 시간·정확도 이점이 함께 확인되지 않았다. 표의 두 값을 각각 해석한다.')
print('각 조건을 한 번씩 실행한 관찰값이므로 작은 차이를 일반화하지 않는다.')

,experiment,max_length,sampling,learning_rate,epochs,val_accuracy,train_seconds,samples_per_second,checkpoint
0,① 파인튜닝 전,142,평가만 수행,NaN,0,0.569115,NaN,NaN,None
1,② 기본 파인튜닝,142,random,0.00002,3,0.908201,4534.6479,77.285,./results/nsmc_step3/checkpoint-29206
2,③ 학습률 변경,142,random,0.00003,3,0.903818,4583.3062,76.464,./results/nsmc_step4/checkpoint-43809
3,④ 버케팅 파인튜닝,142,group_by_length,0.00003,3,0.902859,3161.8328,110.841,./results/nsmc_step5_bucket/checkpoint-43809


STEP 4 대비 학습 시간 절감률: +31.01%
STEP 4 대비 검증 정확도 변화: -0.10 퍼센트포인트
이번 실행에서는 학습 시간이 줄고 정확도가 낮아지는 trade-off(상충관계)가 관찰되었다.
각 조건을 한 번씩 실행한 관찰값이므로 작은 차이를 일반화하지 않는다.


In [18]:
reviews = ['정말 재미있고 다시 보고 싶은 영화다.', '전개가 지루하고 결말도 아쉬웠다.']
inputs = huggingface_tokenizer(reviews, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
inputs = inputs.to(huggingface_model.device)
huggingface_model.eval()
with torch.no_grad():
    predictions = huggingface_model(**inputs).logits.argmax(dim=1).cpu().tolist()
for review, label in zip(reviews, predictions):
    print(review, '→', ['부정', '긍정'][label])

정말 재미있고 다시 보고 싶은 영화다. → 긍정
전개가 지루하고 결말도 아쉬웠다. → 부정


## 루브릭

- 모델과 데이터를 정상적으로 불러오고, 작동하는 것을 확인하였다.
    - klue/bert-base를  NSMC 데이터셋으로 fine-tuning 하여, 모델이 정상적으로 작동하는 것을 확인하였다.

- Preprocessing을 개선하고, fine-tuning을 통해 모델의 성능을 개선시켰다.
    - Validation accuracy를 90% 이상으로 개선하였다.

- 모델 학습에 Bucketing을 성공적으로 적용하고, 그 결과를 비교분석하였다.
    - Bucketing task을 수행하여 fine-tuning 시 연산 속도와 모델 성능 간의 trade-off 관계가 발생하는지 여부를 확인하고, 분석한 결과를 제시하였다.

## 결과 해석과 제출 확인

러닝레이트를 일부 조정해서 파인튜일을 실행했지만, 그 결과는 유의미하게 달라지지 않았다.  

버케팅 역시 학습 시간 단축에는 의미가 있었고, 성능에도 일부 악영향을 끼쳤다.

검증 데이터 29,205개 기준으로 정답 수가 약 28개 줄어들었다. 버케팅으로 배치 구성과 학습 순서가 달라지면서 가중치 업데이트 경로도 달라졌을 가능성이 있다.

| 항목 | ③ 버케팅 미적용 | ④ 버케팅 적용 | 변화 |
|---|---:|---:|---:|
| 검증 정확도 | 90.3818% | 90.2859% | **−0.0959%p** |
| 학습 시간 | 4,583.31초 | 3,161.83초 | **31.01% 단축** |
| 분·초 환산 | 약 76분 23초 | 약 52분 42초 | **약 23분 41초 절약** |
| 초당 처리 샘플 | 76.464개 | 110.841개 | **약 44.96% 증가** |
| 검증 정확도 90% 목표 | 달성 | 달성 | 유지 |